# 06 · Density / size vs F1 analysis

**Purpose:** Quantify how building density and size relate to F1 (per dataset, SpaceNet vs other).

**Inputs:** `outputs/scratch/per_tile_enriched_all_cities.csv`, `aoi_tracker.csv`

**Outputs:** `outputs/scratch/nb06_analysis_summary.md` + figures

**Run order:** After `05`.

**Last run:** _(fill in when you run it)_

> **F1 provenance — IoU threshold**
>
> F1 values in this notebook are **read from pre-computed pipeline outputs**
> (`per_tile_enriched_all_cities.csv` for vector; `raster_metrics_tiles_all_datasets.parquet`
> for raster). The IoU matching threshold applied during pipeline execution is
> **τ = 0.50** (key `iou_threshold` in `configs/validation_configs.yaml`).
>
> To re-run with a different threshold, re-run the full pipeline with the new
> `iou_threshold` value and regenerate `per_tile_enriched_all_cities.csv`
> (notebook 07) before re-running this notebook.
>
> If the pipeline default is later changed (e.g. following the τ=0.25 sensitivity
> test in notebook 10), **this notebook must be re-run** against the new outputs.

In [ ]:
!pip install -q scikit-posthocs statsmodels
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 1 — Setup, load & quality filter ────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.formula.api as smf
import scikit_posthocs as sp
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import yaml

CONFIG_PATH  = Path('/content/drive/MyDrive/urban_validation/configs/validation_configs.yaml')
PROJECT_ROOT = CONFIG_PATH.parents[1]
# Code from GitHub, data from Drive (see colab_bootstrap.py in the repo).
# Drive PROJECT_ROOT/src is a stale hand-copy; never import from it.
import subprocess as _sp
_sp.run(['wget','-q','-O','/content/colab_bootstrap.py','https://raw.githubusercontent.com/GFDRR/urban_validation/fix/pipeline-audit/colab_bootstrap.py'], check=False)
sys.path.insert(0, '/content')
sys.modules.pop('colab_bootstrap', None); assert 'def setup' in open('/content/colab_bootstrap.py').read(), 'bootstrap download failed - check branch/URL'; from colab_bootstrap import setup as _setup
_setup(PROJECT_ROOT)

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
cfg['root_dir'] = str(PROJECT_ROOT)

FIGURES_DIR  = PROJECT_ROOT / 'outputs' / 'figures'
SCRATCH_DIR  = PROJECT_ROOT / 'outputs' / 'scratch'
METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DENSITY_COL = 'ref_building_density_per_km2'
SIZE_COL    = 'mean_ref_building_area_m2'
F1_COL      = 'f1'

# --- Canonical density / size classes (fixed; superseded per-subset quartiles 2026-07-22) ---
# Applied identically to vector AND raster so the two figure families are comparable.
# Building density (buildings per km2) — boundaries loosely aligned with the
# Degree of Urbanisation settlement hierarchy (UN Statistical Commission, 2020)
DENSITY_BREAKS = [0, 20, 100, 300, float('inf')]
DENSITY_LABELS = ['<20', '20–100', '100–300', '>300']
# Mean building footprint size (m2)
SIZE_BREAKS = [0, 80, 120, 180, float('inf')]
SIZE_LABELS = ['<80 m²', '80–120 m²', '120–180 m²', '>180 m²']

# --- Palette, labels & presentation style (Paul Tol Vibrant) ---
import matplotlib.font_manager as fm

# Download Barlow if not cached
import requests, os
font_path = os.path.expanduser('~/.fonts/Barlow-Regular.ttf')
if not os.path.exists(font_path):
    os.makedirs(os.path.dirname(font_path), exist_ok=True)
    url = 'https://github.com/jpt/barlow/raw/main/fonts/ttf/Barlow-Regular.ttf'
    with open(font_path, 'wb') as f:
        f.write(requests.get(url).content)
    fm.fontManager.addfont(font_path)
FONT_FAMILY = 'Barlow'

from src.plots.style import DATASET_COLORS, DATASET_LABELS, apply_ppt_style_mpl as apply_ppt_style  # canonical - keep in sync via src/plots/style.py


def assign_colors(dataset_or_df):
    """Map dataset -> colour from DATASET_COLORS (fallback grey '#AAAAAA').

    Accepts a DataFrame (keys the map by dataset_label via each label's raw
    dataset) or an iterable of raw dataset names.
    """
    if hasattr(dataset_or_df, 'columns'):
        pairs = (dataset_or_df[['dataset', 'dataset_label']]
                 .assign(rawds=lambda d: d['dataset'].astype(str).str.strip().str.lower())
                 .drop_duplicates('dataset_label'))
        return {r.dataset_label: DATASET_COLORS.get(r.rawds, '#AAAAAA')
                for r in pairs.itertuples(index=False)}
    return {name: DATASET_COLORS.get(str(name).strip().lower(), '#AAAAAA')
            for name in dataset_or_df}

# ── Quality filter (applied to both vector and raster) ────────────────────────
def apply_quality_filters(df, label=''):
    """
    Remove tiles that are uninformative or implausible for density/F1 analysis.

    Filters applied (in order):
      DUPLICATE_ROW       — keep first occurrence of (city, dataset, tile_id)
      EMPTY_TILE          — ref_building_count_centroid == 0 (F1 undefined)
      EDGE_TILE           — tile_area_km2 < 0.5 (partial boundary tile)
      TINY_BUILDINGS      — mean_ref_building_area_m2 < 5 m²
      HUGE_BUILDINGS      — mean_ref_building_area_m2 > 10 000 m²
      ALL_ZERO_F1         — all datasets show f1 == 0 for the same (city, tile_id)
      LOW_DENSITY_CITY    — city mean density < 5 bldg/km² (too sparse to analyse)
    """
    n0  = len(df)
    log = []

    # 1. Duplicates — structural data issue, resolve before anything else
    dup = df.duplicated(subset=['city', 'dataset', 'tile_id'], keep='first')
    df  = df[~dup].copy()
    log.append(('DUPLICATE_ROW', int(dup.sum())))

    # 2. Empty tiles — zero reference buildings makes F1 meaningless
    if 'ref_building_count_centroid' in df.columns:
        m  = df['ref_building_count_centroid'] == 0
        df = df[~m].copy()
        log.append(('EMPTY_TILE', int(m.sum())))

    # 3. Edge tiles — partial tiles at AOI boundary inflate density
    if 'tile_area_km2' in df.columns:
        m  = df['tile_area_km2'].notna() & (df['tile_area_km2'] < 0.5)
        df = df[~m].copy()
        log.append(('EDGE_TILE', int(m.sum())))

    # 4. Implausible reference building size
    if SIZE_COL in df.columns:
        m  = df[SIZE_COL].notna() & (df[SIZE_COL] < 5)
        df = df[~m].copy()
        log.append(('TINY_BUILDINGS', int(m.sum())))

        m  = df[SIZE_COL].notna() & (df[SIZE_COL] > 10_000)
        df = df[~m].copy()
        log.append(('HUGE_BUILDINGS', int(m.sum())))

    # 5. All-zero F1 across all datasets for the same tile
    if F1_COL in df.columns:
        all_zero = df.groupby(['city', 'tile_id'])[F1_COL].transform(
            lambda x: (x == 0).all()
        )
        m  = all_zero.astype(bool)
        df = df[~m].copy()
        log.append(('ALL_ZERO_F1', int(m.sum())))

    # 6. Low-density cities — entire city excluded (too sparse, density analysis unreliable)
    if DENSITY_COL in df.columns:
        city_density  = df.groupby('city')[DENSITY_COL].mean()
        low_cities    = city_density[city_density < 5].index
        m             = df['city'].isin(low_cities)
        df            = df[~m].copy()
        log.append(('LOW_DENSITY_CITY (<5 bldg/km²)', int(m.sum())))
        if len(low_cities):
            print(f'  [{label}] Excluded low-density cities: {sorted(low_cities.tolist())}')

    # Summary
    n_removed = n0 - len(df)
    pct_kept  = len(df) / n0 * 100 if n0 else 0
    print(f'\n  [{label}] Quality filter: {n0:,} → {len(df):,} rows  ({pct_kept:.1f}% retained)')
    for flag, n in log:
        if n:
            print(f'    {flag:<35}: {n:>5,} rows removed')
    return df

# ── Load vector enriched CSV ──────────────────────────────────────────────────
DS_LABELS_VEC = DATASET_LABELS

df_vec = pd.read_csv(SCRATCH_DIR / 'per_tile_enriched_all_cities.csv')
df_vec['dataset_label'] = df_vec['dataset'].str.lower().map(DS_LABELS_VEC).fillna(df_vec['dataset'])

print(f'Loaded: {len(df_vec):,} rows | {df_vec["city"].nunique()} cities')

df_vec       = apply_quality_filters(df_vec, 'vector')
df_vec_clean = df_vec.dropna(subset=[DENSITY_COL, SIZE_COL, F1_COL]).copy()

print(f'\n=== Vector working set ===')
print(f'  Rows   : {len(df_vec_clean):,}')
print(f'  Cities : {df_vec_clean["city"].nunique()}')
print(f'  Datasets: {sorted(df_vec_clean["dataset"].str.lower().unique())}')

In [ ]:
# ── Zero-F1 city exclusion (item 8b) ─────────────────────────────────────────
# Drop tiles belonging to cities that FAIL the vector city-level filter: cities
# where f1_city == 0 for overture AND gba AND globfp in vector_all_cities_merged.csv.
# These are coverage failures / pipeline errors, not genuine zero-accuracy results,
# and they distort global means and error bars. In-memory only — the raw
# per_tile_enriched_all_cities.csv is NOT modified.
#
# The excluded-city list is COMPUTED AT RUNTIME from the city-level merged CSV
# (no city names are hard-coded), so it matches the filter used in notebooks
# 06/07 exactly. Such a city also has f1 == 0 on every tile, so most of its tiles
# are already dropped by the ALL_ZERO_F1 tile filter above; this step makes the
# city-level exclusion explicit and reports which cities are affected.

VECTOR_FILTER_DATASETS = ['overture', 'gba', 'globfp']
_merged_vec_path = PROJECT_ROOT / 'outputs' / 'global_metrics' / 'vector_all_cities_merged.csv'

excluded_cities = []
if _merged_vec_path.exists():
    _mv = pd.read_csv(_merged_vec_path)
    _mv['_ds'] = _mv['dataset'].astype(str).str.strip().str.lower()
    _mv['f1_city'] = pd.to_numeric(_mv['f1_city'], errors='coerce')
    # Pivot to city x dataset; a city is excluded only if all three datasets are
    # present and every one is 0 (a missing dataset -> NaN -> NaN == 0 is False).
    _vpiv = (_mv.drop_duplicates(['city', '_ds'])
                .pivot(index='city', columns='_ds', values='f1_city')
                .reindex(columns=VECTOR_FILTER_DATASETS))
    excluded_cities = sorted(_vpiv.index[(_vpiv == 0).all(axis=1)].tolist())
else:
    print(f"[WARN] {_merged_vec_path} not found — city-level zero-F1 filter skipped "
          f"(tile-level ALL_ZERO_F1 filter above still applies).")

if excluded_cities:
    print(f"Excluding {len(excluded_cities)} cities (all vector datasets F1=0): {excluded_cities}")
else:
    print("No cities excluded (no city has all vector datasets F1=0).")

# Filter in place (reassign) — the raw tile frame and the analysis working set.
df_vec       = df_vec[~df_vec['city'].isin(excluded_cities)]
df_vec_clean = df_vec_clean[~df_vec_clean['city'].isin(excluded_cities)]
print(f"Vector cities remaining: {df_vec_clean['city'].nunique()}  ({len(df_vec_clean)} tiles)")

In [ ]:
# ── Cell 2 — Reference-source detection (SpaceNet vs other) ──────────────────
# Reads `reference_source` column from the AOI tracker CSV.
# Values: 'spacenet' | 'other'  (set by the aoi_tracker update script).
# Falls back to filename heuristic if the column is absent.

tracker_path = PROJECT_ROOT / 'data/02_interim/aoi_tracker.csv'
if 'cfg' in dir() and cfg.get('aoi_tracker'):
    p = Path(cfg['aoi_tracker'])
    tracker_path = p if p.is_absolute() else PROJECT_ROOT / p

ref_source_map = {}  # city_id -> 'spacenet' | 'other'

if tracker_path.exists():
    # utf-8-sig strips BOM (﻿) written by Excel or Windows tools
    tracker = pd.read_csv(tracker_path, dtype=str, encoding='utf-8-sig')
    # Normalise column names: strip whitespace + BOM + zero-width spaces
    tracker.columns = (tracker.columns
                       .str.strip()
                       .str.replace('﻿', '', regex=False)
                       .str.replace('​', '', regex=False))
    id_col = 'dataset_folder_name'

    print(f'Tracker path  : {tracker_path}')
    print(f'Tracker shape : {tracker.shape}')
    print(f'Last 5 columns: {tracker.columns.tolist()[-5:]}')

    if 'reference_source' in tracker.columns and id_col in tracker.columns:
        ref_source_map = (
            tracker.dropna(subset=[id_col])
            .set_index(id_col)['reference_source']
            .str.strip().str.lower()
            .to_dict()
        )
        n_sn_tracker = sum(1 for v in ref_source_map.values() if v == 'spacenet')
        print(f'reference_source column found: {len(ref_source_map)} cities, {n_sn_tracker} SpaceNet')

    elif id_col in tracker.columns:
        ref_col = next((c for c in tracker.columns
                        if 'reference' in c.lower() and 'file' in c.lower()), None)
        if ref_col:
            city_ref = (
                tracker.groupby(id_col)[ref_col]
                .apply(lambda x: '|'.join(x.dropna().str.lower()))
                .reset_index()
                .rename(columns={id_col: 'city', ref_col: 'ref_files'})
            )
            city_ref['reference_source'] = city_ref['ref_files'].apply(
                lambda x: 'spacenet' if 'spacenet' in str(x) else 'other'
            )
            ref_source_map = city_ref.set_index('city')['reference_source'].to_dict()
            print(f'[WARN] reference_source column not found — inferred from filename')
            print('       Run the aoi_tracker update script to add the column permanently.')
        else:
            print('[WARN] Neither reference_source nor reference file column found.')
    else:
        print(f'[WARN] id_col "{id_col}" not in tracker. Available: {tracker.columns.tolist()}')
else:
    print(f'[WARN] Tracker not found at {tracker_path} — defaulting all to "other".')

def tag_ref_source(df):
    df = df.copy()
    df['ref_source'] = df['city'].map(ref_source_map).fillna('other')
    return df

df_vec_clean = tag_ref_source(df_vec_clean)

n_sn  = df_vec_clean['city'][df_vec_clean['ref_source'] == 'spacenet'].nunique()
n_oth = df_vec_clean['city'][df_vec_clean['ref_source'] == 'other'].nunique()
print(f'\n  SpaceNet cities : {n_sn}')
print(f'  Other cities    : {n_oth}')
if n_sn:
    sn_list = sorted(df_vec_clean[df_vec_clean['ref_source'] == 'spacenet']['city'].unique())
    print(f'  SpaceNet city list : {sn_list}')

def make_subsets(df):
    return {
        'all':          df,
        'non_spacenet': df[df['ref_source'] == 'other'].copy(),
        'spacenet':     df[df['ref_source'] == 'spacenet'].copy(),
    }

VEC_SUBSETS = make_subsets(df_vec_clean)
print('\n  Subset sizes (vector):')
for k, v in VEC_SUBSETS.items():
    print(f'    {k:<15}: {len(v):>6,} rows | {v["city"].nunique()} cities')

In [ ]:
# ── Cell 3 — Analysis helper functions ───────────────────────────────────────
# Defined once; called for every (data_type × subset) combination.

# superseded 2026-07-22 — data-driven quartiles; no longer called (kept for provenance)
def global_quartile_bins(df, col=DENSITY_COL):
    """Compute quartile boundaries from the full dataset.
    Applying the same bins to subsets keeps Q1-Q4 definitions consistent.
    """
    q = df[col].quantile([0, 0.25, 0.5, 0.75, 1.0]).values
    labels = [
        f'Q1\n(<{q[1]:.0f})',
        f'Q2\n({q[1]:.0f}–{q[2]:.0f})',
        f'Q3\n({q[2]:.0f}–{q[3]:.0f})',
        f'Q4\n(>{q[3]:.0f})',
    ]
    return q, labels


def add_density_q(df, bins, labels):
    df = df.copy()
    df['density_q'] = pd.cut(df[DENSITY_COL], bins=bins, labels=labels, include_lowest=True)
    return df


def add_size_q(df, bins, labels):
    df = df.copy()
    df['size_q'] = pd.cut(df[SIZE_COL], bins=bins, labels=labels, include_lowest=True)
    return df


def _section_header(title, group_label, n_rows, n_cities):
    bar = '─' * 60
    print(f'\n{bar}')
    print(f'  {title}  |  group: {group_label}  ({n_rows:,} rows, {n_cities} cities)')
    print(bar)


def run_spearman(df, group_label):
    """Spearman ρ per dataset for density→F1 and size→F1."""
    _section_header('Spearman ρ (marginal)', group_label, len(df), df['city'].nunique())
    results = []
    for ds in sorted(df['dataset'].str.lower().unique()):
        sub = df[df['dataset'].str.lower() == ds]
        label = DS_LABELS_VEC.get(ds, ds)
        rho_d, p_d = stats.spearmanr(sub[DENSITY_COL], sub[F1_COL])
        rho_s, p_s = stats.spearmanr(sub[SIZE_COL],    sub[F1_COL])
        sig_d = '***' if p_d < 0.001 else ('**' if p_d < 0.01 else ('*' if p_d < 0.05 else 'ns'))
        sig_s = '***' if p_s < 0.001 else ('**' if p_s < 0.01 else ('*' if p_s < 0.05 else 'ns'))
        print(f'{label} (n={len(sub):,})')
        print(f'  density→F1: ρ = {rho_d:+.3f}  p = {p_d:.2e}  {sig_d}')
        print(f'  size→F1:    ρ = {rho_s:+.3f}  p = {p_s:.2e}  {sig_s}')
        results.append(dict(group=group_label, dataset=label, n=len(sub),
                            rho_density=rho_d, p_density=p_d,
                            rho_size=rho_s, p_size=p_s))
    return results


def run_kw(df, group_label, q_labels,
           q_col='density_q',
           title='Kruskal–Wallis + Dunn (Bonferroni) — density classes'):
    """Kruskal–Wallis + Dunn post-hoc per dataset for the given class column."""
    _section_header(title, group_label, len(df), df['city'].nunique())
    results = []
    for ds in sorted(df['dataset'].str.lower().unique()):
        sub   = df[df['dataset'].str.lower() == ds].dropna(subset=[q_col])
        label = DS_LABELS_VEC.get(ds, ds)
        groups = [g[F1_COL].values for _, g in sub.groupby(q_col, observed=True) if len(g) > 0]
        if len(groups) < 2:
            print(f'{label}: insufficient groups — skipped')
            continue
        H, p = stats.kruskal(*groups)
        sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        print(f'{label}: H = {H:.2f}  p = {p:.2e}  {sig}')
        if p < 0.05:
            dunn = sp.posthoc_dunn(sub, val_col=F1_COL, group_col=q_col, p_adjust='bonferroni')
            print(dunn.round(4).to_string())
        results.append(dict(group=group_label, dataset=label, H=H, p=p, sig=sig))
    return results


def plot_boxplot(df, group_label, q_labels, color_map, save_tag, figures_dir,
                 q_col='density_q',
                 xlabel='Density class (bldg/km²)',
                 file_prefix='f1_by_density_class'):
    """Box plot: F1 by class (density or size), one panel per dataset."""
    ds_order = [d for d in sorted(color_map) if d in df['dataset_label'].unique()]
    if not ds_order:
        print(f'  [SKIP] no datasets for boxplot ({group_label})')
        return
    fig, axes = plt.subplots(1, len(ds_order), figsize=(4.5 * len(ds_order), 5), sharey=True)
    if len(ds_order) == 1:
        axes = [axes]
    for ax, ds_label in zip(axes, ds_order):
        sub   = df[df['dataset_label'] == ds_label].dropna(subset=[q_col])
        color = color_map.get(ds_label, '#AAAAAA')
        sns.boxplot(data=sub, x=q_col, y=F1_COL, ax=ax, order=q_labels,
                    color=color, width=0.55, linewidth=1.2,
                    flierprops=dict(marker='o', markersize=2, alpha=0.25,
                                   markerfacecolor=color, markeredgecolor='none'),
                    medianprops=dict(color='white', linewidth=2))
        for i, ql in enumerate(q_labels):
            n = (sub[q_col] == ql).sum()
            ax.text(i, -0.06, f'n={n:,}', ha='center', va='top', fontsize=7.5,
                    color='#555555', transform=ax.get_xaxis_transform())
        ax.set_title(ds_label, fontsize=12, fontweight='bold', color=color, pad=8)
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel('F1 score' if ax == axes[0] else '', fontsize=9)
        ax.set_ylim(-0.05, 1.05)
        ax.yaxis.set_major_locator(mticker.MultipleLocator(0.2))
        ax.grid(axis='y', color='#e0e0e0', linewidth=0.7, zorder=0)
        ax.set_axisbelow(True)
        sns.despine(ax=ax)
    title_var = xlabel.split('(')[0].strip()
    fig.suptitle(f'F1 by {title_var.lower()} — {group_label}', fontsize=13, y=1.01)
    fig.tight_layout()
    apply_ppt_style(fig, axes)
    path = figures_dir / f'{file_prefix}_{save_tag}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved → {path}')


def plot_scatters(df, group_label, color_map, save_tag, figures_dir):
    """Scatter: (a) density vs F1 (log-x), (b) mean size vs F1. Coloured by dataset."""
    ds_order = [d for d in sorted(color_map) if d in df['dataset_label'].unique()]

    def _scatter(ax, x_col, x_label, log_x=False):
        for ds_label in ds_order:
            sub = df[df['dataset_label'] == ds_label]
            ax.scatter(sub[x_col], sub[F1_COL], color=color_map.get(ds_label, '#AAAAAA'),
                       alpha=0.3, s=8, linewidths=0, label=ds_label, rasterized=True)
        if log_x:
            ax.set_xscale('log')
        ax.set_xlabel(x_label, fontsize=10)
        ax.set_ylabel('F1 score', fontsize=10)
        ax.set_ylim(-0.02, 1.05)
        ax.grid(color='#e8e8e8', linewidth=0.6, zorder=0)
        ax.set_axisbelow(True)
        ax.legend(title='Dataset', fontsize=8, title_fontsize=8,
                  markerscale=3, framealpha=0.85)
        sns.despine(ax=ax)

    for x_col, x_label, log_x, fname in [
        (DENSITY_COL, 'Reference building density (bldg/km²)', True,  f'density_f1_scatter_{save_tag}.png'),
        (SIZE_COL,    'Mean reference building area (m²)',      False, f'size_f1_scatter_{save_tag}.png'),
    ]:
        fig, ax = plt.subplots(figsize=(7, 5))
        _scatter(ax, x_col, x_label, log_x)
        ax.set_title(f'{x_label.split("(")[0].strip()} vs F1 — {group_label}', fontsize=12)
        fig.tight_layout()
        apply_ppt_style(fig, ax)
        path = figures_dir / fname
        fig.savefig(path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'  Saved → {path}')


def run_lme(df, group_label):
    """Linear mixed-effects model: f1 ~ density_z + size_z + C(dataset), city random intercept."""
    _section_header('Linear mixed-effects model', group_label, len(df), df['city'].nunique())
    dfl = df[['city', 'dataset', F1_COL, DENSITY_COL, SIZE_COL]].dropna().copy()
    dfl['density_z'] = (dfl[DENSITY_COL] - dfl[DENSITY_COL].mean()) / dfl[DENSITY_COL].std()
    dfl['size_z']    = (dfl[SIZE_COL]    - dfl[SIZE_COL].mean())    / dfl[SIZE_COL].std()
    dfl['dataset']   = dfl['dataset'].str.lower()
    if dfl['city'].nunique() < 2:
        print('  [SKIP] fewer than 2 cities — LME cannot fit random intercept')
        return None
    result = smf.mixedlm('f1 ~ density_z + size_z + C(dataset)', data=dfl,
                         groups=dfl['city']).fit(reml=True)
    print(result.summary())
    print('\n  Key fixed effects:')
    for name, label in [('density_z', 'density (z)'), ('size_z', 'mean size (z)')]:
        coef = result.params[name]
        pval = result.pvalues[name]
        ci   = result.conf_int().loc[name]
        sig  = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'ns'))
        print(f'  {label:<20}: β = {coef:+.4f}  [{ci[0]:+.4f}, {ci[1]:+.4f}]  p = {pval:.2e}  {sig}')
    print(f'  Random-effect var (city): {result.cov_re.iloc[0,0]:.4f}')
    return result


def run_group(df, group_label,
              q_bins, q_labels,
              sq_bins, sq_labels,
              color_map, save_tag, figures_dir):
    """Orchestrate all analyses for one (data_type, subset) combination."""
    print(f'\n{"="*65}')
    print(f'  GROUP: {group_label}  ({len(df):,} rows | {df["city"].nunique()} cities)')
    print(f'{"="*65}')
    if len(df) < 10:
        print('  [SKIP] fewer than 10 rows')
        return {}

    df_q = add_density_q(df, q_bins, q_labels)
    df_q = add_size_q(df_q, sq_bins, sq_labels)

    # Spearman (computes both density and size in one pass)
    s = run_spearman(df_q, group_label)

    # Kruskal–Wallis: density classes
    kw = run_kw(df_q, group_label, q_labels,
                q_col='density_q',
                title='Kruskal–Wallis + Dunn (Bonferroni) — density classes')

    # Kruskal–Wallis: size classes
    kw_size = run_kw(df_q, group_label, sq_labels,
                     q_col='size_q',
                     title='Kruskal–Wallis + Dunn (Bonferroni) — size classes')

    # Box plots: density classes
    plot_boxplot(df_q, group_label, q_labels, color_map, save_tag, figures_dir,
                 q_col='density_q',
                 xlabel='Density class (bldg/km²)',
                 file_prefix='f1_by_density_class')

    # Box plots: size classes
    plot_boxplot(df_q, group_label, sq_labels, color_map, save_tag, figures_dir,
                 q_col='size_q',
                 xlabel='Size class (m²)',
                 file_prefix='f1_by_size_class')

    plot_scatters(df_q, group_label, color_map, save_tag, figures_dir)
    lme = run_lme(df_q, group_label)
    return dict(spearman=s, kw=kw, kw_size=kw_size, lme=lme)


print('Analysis functions defined.')

---
## Vector analysis
Candidates: Overture, GBA, GlobFP — compared against local reference buildings.

In [ ]:
# ── Cell 4 — Vector analysis (all 3 subsets) ──────────────────────────────────
# superseded 2026-07-22: global_quartile_bins(df_vec_clean, col=DENSITY_COL)  # per-subset quartiles
VEC_Q_BINS,  VEC_Q_LABELS  = DENSITY_BREAKS, DENSITY_LABELS
# superseded 2026-07-22: global_quartile_bins(df_vec_clean, col=SIZE_COL)  # per-subset quartiles
VEC_SQ_BINS, VEC_SQ_LABELS = SIZE_BREAKS, SIZE_LABELS
VEC_DS_ALL    = sorted(df_vec_clean['dataset_label'].unique())
VEC_COLOR_MAP = assign_colors(df_vec_clean)

print('Fixed vector density class boundaries (bldg/km²):')
for lab, (lo, hi) in zip(VEC_Q_LABELS, zip(VEC_Q_BINS[:-1], VEC_Q_BINS[1:])):
    print(f'  {lab}: {lo:.0f} – {hi:.0f}')

print('\nFixed vector size class boundaries (m²):')
for lab, (lo, hi) in zip(VEC_SQ_LABELS, zip(VEC_SQ_BINS[:-1], VEC_SQ_BINS[1:])):
    print(f'  {lab}: {lo:.0f} – {hi:.0f}')

print(f'\nDataset colour map: {VEC_COLOR_MAP}')

vec_results = {}
for group_label, sdf in VEC_SUBSETS.items():
    save_tag = f'vector_{group_label}'
    vec_results[group_label] = run_group(
        sdf, group_label,
        VEC_Q_BINS,  VEC_Q_LABELS,
        VEC_SQ_BINS, VEC_SQ_LABELS,
        VEC_COLOR_MAP, save_tag, FIGURES_DIR,
    )

---
## Raster analysis
Raster candidates (GHSL, WSF, …) compared against the same reference buildings.
Density columns are joined from the vector enriched CSV (same tile grid).

In [ ]:
# ── Cell 5 — Load raster data, join density & quality filter ──────────────────
RASTER_SENTINEL = 'raster_metrics_tiles_all_datasets.parquet'

# RASTER_GRID: set to a specific grid name (e.g. 'native') to filter to one
# evaluation resolution, or None to average f1 across all grids per
# (city, tile_id, dataset).
RASTER_GRID = None

raster_parts    = []
raster_city_dirs = sorted(p.parent for p in METRICS_ROOT.rglob(RASTER_SENTINEL))
print(f'Found {len(raster_city_dirs)} cities with raster metrics')

for city_dir in raster_city_dirs:
    try:
        part = pd.read_parquet(city_dir / RASTER_SENTINEL)
        raster_parts.append(part)
    except Exception as e:
        print(f'  [WARN] {city_dir.name}: {e}')

if not raster_parts:
    print('[INFO] No raster metrics found — raster analysis will be skipped.')
    df_rast_clean = pd.DataFrame(columns=['city', 'tile_id', 'dataset', 'dataset_label',
                                           F1_COL, DENSITY_COL, SIZE_COL, 'ref_source'])
else:
    df_rast = pd.concat(raster_parts, ignore_index=True)
    print(f'Raster rows loaded : {len(df_rast):,}')

    # sorted() fails on mixed str/NaN — drop NaN before sorting
    if 'grid' in df_rast.columns:
        grids_found = sorted(df_rast['grid'].dropna().unique().astype(str))
    else:
        grids_found = []
    print(f'Evaluation grids   : {grids_found or "no grid column"}')
    print(f'Raster datasets    : {sorted(df_rast["dataset"].dropna().unique())}')

    # Filter or aggregate across evaluation grids
    if 'grid' in df_rast.columns:
        if RASTER_GRID is not None:
            df_rast = df_rast[df_rast['grid'] == RASTER_GRID].copy()
            print(f'Filtered to grid "{RASTER_GRID}": {len(df_rast):,} rows')
        else:
            agg_cols = {k: 'mean' for k in [F1_COL, 'precision', 'recall'] if k in df_rast.columns}
            keep_cols = ['city', 'tile_id', 'dataset'] + list(agg_cols)
            df_rast = (df_rast[keep_cols]
                       .groupby(['city', 'tile_id', 'dataset'])
                       .agg(agg_cols)
                       .reset_index())
            print(f'Averaged across grids: {len(df_rast):,} rows')

    # Join density columns from vector enriched CSV (same tile grid)
    density_lookup = (
        df_vec_clean[['city', 'tile_id', DENSITY_COL, SIZE_COL,
                       'tile_area_km2', 'ref_building_count_centroid']]
        .drop_duplicates(subset=['city', 'tile_id'])
    )
    df_rast = df_rast.merge(density_lookup, on=['city', 'tile_id'], how='left')
    df_rast['dataset_label'] = df_rast['dataset'].astype(str).str.strip().str.lower().map(DATASET_LABELS).fillna(df_rast['dataset'])

    n_pre_filter = len(df_rast)
    df_rast = apply_quality_filters(df_rast, 'raster')
    df_rast_clean = df_rast.dropna(subset=[DENSITY_COL, SIZE_COL, F1_COL]).copy()
    df_rast_clean = tag_ref_source(df_rast_clean)

    joined_pct = len(df_rast_clean) / n_pre_filter * 100 if n_pre_filter else 0
    print(f'\nRaster working set: {len(df_rast_clean):,} rows ({joined_pct:.1f}% of pre-filter)')
    print(f'Cities: {df_rast_clean["city"].nunique()}')
    if joined_pct < 50:
        print('[WARN] > 50% of raster rows dropped — many cities may lack density'
              ' enrichment (missing tiles GPKG). Results cover a partial sample.')

In [ ]:
# ── Cell 6 — Raster analysis (all 3 subsets) ──────────────────────────────────
if df_rast_clean.empty:
    print('No raster data — skipping.')
else:
    # superseded 2026-07-22: global_quartile_bins(df_rast_clean, col=DENSITY_COL)  # per-subset quartiles
    RAST_Q_BINS,  RAST_Q_LABELS  = DENSITY_BREAKS, DENSITY_LABELS
    # superseded 2026-07-22: global_quartile_bins(df_rast_clean, col=SIZE_COL)  # per-subset quartiles
    RAST_SQ_BINS, RAST_SQ_LABELS = SIZE_BREAKS, SIZE_LABELS
    RAST_DS_ALL    = sorted(df_rast_clean['dataset_label'].unique())
    RAST_COLOR_MAP = assign_colors(df_rast_clean)
    RAST_SUBSETS   = make_subsets(df_rast_clean)

    print('Fixed raster density class boundaries (bldg/km²):')
    for lab, (lo, hi) in zip(RAST_Q_LABELS, zip(RAST_Q_BINS[:-1], RAST_Q_BINS[1:])):
        print(f'  {lab}: {lo:.0f} – {hi:.0f}')

    print('\nFixed raster size class boundaries (m²):')
    for lab, (lo, hi) in zip(RAST_SQ_LABELS, zip(RAST_SQ_BINS[:-1], RAST_SQ_BINS[1:])):
        print(f'  {lab}: {lo:.0f} – {hi:.0f}')

    print(f'\nRaster dataset colour map: {RAST_COLOR_MAP}')
    print('\n  Subset sizes (raster):')
    for k, v in RAST_SUBSETS.items():
        print(f'    {k:<15}: {len(v):>6,} rows | {v["city"].nunique()} cities')

    rast_results = {}
    for group_label, sdf in RAST_SUBSETS.items():
        save_tag = f'raster_{group_label}'
        rast_results[group_label] = run_group(
            sdf, group_label,
            RAST_Q_BINS,  RAST_Q_LABELS,
            RAST_SQ_BINS, RAST_SQ_LABELS,
            RAST_COLOR_MAP, save_tag, FIGURES_DIR,
        )

In [ ]:
# ── Cell 7 — Shareable summary report ────────────────────────────────────────
# Collects all results into a markdown file a colleague can read without
# needing to open the notebook.

from datetime import date

SUMMARY_PATH = SCRATCH_DIR / 'nb06_analysis_summary.md'

def sig_stars(p):
    if p is None or (hasattr(p, '__float__') and np.isnan(float(p))):
        return 'n/a'
    return '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))

lines = []
L = lines.append

L(f'# Notebook 06 — Building density / size vs F1 analysis')
L(f'**Generated:** {date.today().isoformat()}  ')
L(f'**Source notebook:** `06_analysis_density_f1.ipynb` on branch `fix/pipeline-audit`\n')

# ── Data & filters ─────────────────────────────────────────────────────────────
L('---')
L('## 1. Input data\n')
L('### Vector (Overture, GBA, GlobFP vs local reference buildings)')
L(f'- Cities: **{df_vec_clean["city"].nunique()}**')
L(f'- Tiles × datasets: **{len(df_vec_clean):,}** rows')
L(f'- Candidate datasets: {", ".join(sorted(df_vec_clean["dataset_label"].unique()))}')
L(f'- Reference density range: {df_vec_clean[DENSITY_COL].min():.1f} – {df_vec_clean[DENSITY_COL].max():.1f} bldg/km²\n')

if 'df_rast_clean' in dir() and not df_rast_clean.empty:
    L('### Raster candidates')
    L(f'- Cities: **{df_rast_clean["city"].nunique()}**')
    L(f'- Tiles × datasets: **{len(df_rast_clean):,}** rows')
    L(f'- Candidate datasets: {", ".join(sorted(df_rast_clean["dataset_label"].unique()))}\n')
else:
    L('### Raster candidates\n- No raster data available.\n')

L('### Quality filters applied (auto, before analysis)')
L('| Filter | Condition |')
L('|---|---|')
L('| DUPLICATE_ROW | duplicate (city, dataset, tile_id) — keep first |')
L('| EMPTY_TILE | ref_building_count_centroid == 0 |')
L('| EDGE_TILE | tile_area_km2 < 0.5 km² |')
L('| TINY_BUILDINGS | mean_ref_building_area_m2 < 5 m² |')
L('| HUGE_BUILDINGS | mean_ref_building_area_m2 > 10 000 m² |')
L('| ALL_ZERO_F1 | all datasets show F1 = 0 for the same tile |')
L('| LOW_DENSITY_CITY | city mean density < 5 bldg/km² (entire city excluded) |\n')

# ── SpaceNet split ─────────────────────────────────────────────────────────────
L('---')
L('## 2. Reference dataset split\n')
sn_cities  = sorted(df_vec_clean[df_vec_clean['ref_source'] == 'spacenet']['city'].unique())
oth_cities = sorted(df_vec_clean[df_vec_clean['ref_source'] == 'other']['city'].unique())
L(f'- **SpaceNet reference** ({len(sn_cities)} cities): {", ".join(sn_cities) if sn_cities else "none detected"}')
L(f'- **Other reference** ({len(oth_cities)} cities): detected from AOI tracker CSV')
L('')
L('> SpaceNet vs other is read from the `reference_source` column of `aoi_tracker.csv`.')
L('> This is the same column that sets the per-city IoU threshold in the vector pipeline.\n')

# ── Density class boundaries ────────────────────────────────────────────────
L('---')
L('## 3. Density class boundaries (fixed, applied to all groups)\n')
L('Fixed canonical classes (superseded per-subset quartiles 2026-07-22), applied identically to vector and raster so the figure families are comparable.\n')
L('| Density class | Range (bldg/km²) |')
L('|---|---|')
for lab, (lo, hi) in zip(VEC_Q_LABELS, zip(VEC_Q_BINS[:-1], VEC_Q_BINS[1:])):
    L(f'| {lab} | {lo:.0f} – {hi:.0f} |')
L('')

# ── Vector density results ─────────────────────────────────────────────────────
L('---')
L('## 4. Vector density results\n')

for group_label, res in vec_results.items():
    if not res:
        continue
    L(f'### {group_label.replace("_", " ").title()}\n')

    if res.get('spearman'):
        L('**Spearman ρ — density→F1** (marginal — tiles nested in cities)\n')
        L('| Dataset | n tiles | ρ | p |')
        L('|---|---|---|---|')
        for r in res['spearman']:
            L(f'| {r["dataset"]} | {r["n"]:,} | {r["rho_density"]:+.3f} | {sig_stars(r["p_density"])} ({r["p_density"]:.2e}) |')
        L('')

    if res.get('kw'):
        L('**Kruskal–Wallis** (F1 across density classes, Dunn+Bonferroni post-hoc if p < 0.05)\n')
        L('| Dataset | H | p | Significant? |')
        L('|---|---|---|---|')
        for r in res['kw']:
            L(f'| {r["dataset"]} | {r["H"]:.2f} | {r["p"]:.2e} | {r["sig"]} |')
        L('')

    if res.get('lme') is not None:
        result = res['lme']
        L('**Linear mixed-effects model** (city random intercept; β per 1-SD increase)\n')
        L('| Predictor | β | 95% CI | p |')
        L('|---|---|---|---|')
        for name, lbl in [('density_z', 'Density (z-score)'), ('size_z', 'Mean size (z-score)')]:
            try:
                coef = result.params[name]
                pval = result.pvalues[name]
                ci   = result.conf_int().loc[name]
                L(f'| {lbl} | {coef:+.4f} | [{ci[0]:+.4f}, {ci[1]:+.4f}] | {sig_stars(pval)} ({pval:.2e}) |')
            except KeyError:
                pass
        try:
            L(f'| Random-effect variance (city) | {result.cov_re.iloc[0,0]:.4f} | — | — |')
        except Exception:
            pass
        L('')

# ── Vector building size results ───────────────────────────────────────────────
L('---')
L('## 5. Vector building size results\n')

L('### Size class boundaries (fixed, applied to all groups)\n')
L('Fixed canonical classes (same logic as the density classes).\n')
L('| Size class | Range (m²) |')
L('|---|---|')
for lab, (lo, hi) in zip(VEC_SQ_LABELS, zip(VEC_SQ_BINS[:-1], VEC_SQ_BINS[1:])):
    L(f'| {lab} | {lo:.0f} – {hi:.0f} |')
L('')

for group_label, res in vec_results.items():
    if not res:
        continue
    L(f'### {group_label.replace("_", " ").title()}\n')

    if res.get('spearman'):
        L('**Spearman ρ — size→F1** (marginal)\n')
        L('| Dataset | n tiles | ρ | p |')
        L('|---|---|---|---|')
        for r in res['spearman']:
            L(f'| {r["dataset"]} | {r["n"]:,} | {r["rho_size"]:+.3f} | {sig_stars(r["p_size"])} ({r["p_size"]:.2e}) |')
        L('')

    if res.get('kw_size'):
        L('**Kruskal–Wallis** (F1 across size classes, Dunn+Bonferroni post-hoc if p < 0.05)\n')
        L('| Dataset | H | p | Significant? |')
        L('|---|---|---|---|')
        for r in res['kw_size']:
            L(f'| {r["dataset"]} | {r["H"]:.2f} | {r["p"]:.2e} | {r["sig"]} |')
        L('')

# ── Raster density results ─────────────────────────────────────────────────────
if 'rast_results' in dir() and rast_results:
    L('---')
    L('## 6. Raster density results\n')
    for group_label, res in rast_results.items():
        if not res:
            continue
        L(f'### {group_label.replace("_", " ").title()}\n')
        if res.get('spearman'):
            L('**Spearman ρ — density→F1**\n')
            L('| Dataset | n tiles | ρ | p |')
            L('|---|---|---|---|')
            for r in res['spearman']:
                L(f'| {r["dataset"]} | {r["n"]:,} | {r["rho_density"]:+.3f} | {sig_stars(r["p_density"])} ({r["p_density"]:.2e}) |')
            L('')
        if res.get('kw'):
            L('**Kruskal–Wallis** (density classes)\n')
            L('| Dataset | H | p | Significant? |')
            L('|---|---|---|---|')
            for r in res['kw']:
                L(f'| {r["dataset"]} | {r["H"]:.2f} | {r["p"]:.2e} | {r["sig"]} |')
            L('')
        if res.get('lme') is not None:
            result = res['lme']
            L('**LME fixed effects**\n')
            L('| Predictor | β | 95% CI | p |')
            L('|---|---|---|---|')
            for name, lbl in [('density_z', 'Density (z-score)'), ('size_z', 'Mean size (z-score)')]:
                try:
                    coef = result.params[name]
                    pval = result.pvalues[name]
                    ci   = result.conf_int().loc[name]
                    L(f'| {lbl} | {coef:+.4f} | [{ci[0]:+.4f}, {ci[1]:+.4f}] | {sig_stars(pval)} ({pval:.2e}) |')
                except KeyError:
                    pass
            L('')

    # ── Raster building size results ───────────────────────────────────────────
    L('---')
    L('## 7. Raster building size results\n')

    L('### Size class boundaries (fixed, applied to all raster groups)\n')
    L('| Size class | Range (m²) |')
    L('|---|---|')
    for lab, (lo, hi) in zip(RAST_SQ_LABELS, zip(RAST_SQ_BINS[:-1], RAST_SQ_BINS[1:])):
        L(f'| {lab} | {lo:.0f} – {hi:.0f} |')
    L('')

    for group_label, res in rast_results.items():
        if not res:
            continue
        L(f'### {group_label.replace("_", " ").title()}\n')
        if res.get('spearman'):
            L('**Spearman ρ — size→F1**\n')
            L('| Dataset | n tiles | ρ | p |')
            L('|---|---|---|---|')
            for r in res['spearman']:
                L(f'| {r["dataset"]} | {r["n"]:,} | {r["rho_size"]:+.3f} | {sig_stars(r["p_size"])} ({r["p_size"]:.2e}) |')
            L('')
        if res.get('kw_size'):
            L('**Kruskal–Wallis** (size classes)\n')
            L('| Dataset | H | p | Significant? |')
            L('|---|---|---|---|')
            for r in res['kw_size']:
                L(f'| {r["dataset"]} | {r["H"]:.2f} | {r["p"]:.2e} | {r["sig"]} |')
            L('')

# ── Caveats ────────────────────────────────────────────────────────────────────
L('---')
L('## 8. Caveats & interpretation notes\n')
L('1. **Spearman ρ is marginal.** Tiles are nested within cities. City-level confounding')
L('   (e.g. a SpaceNet city with high density everywhere) inflates or deflates ρ. Use the')
L('   LME coefficient as the primary effect-size measure — it controls for city.')
L('')
L('2. **Coverage.** The analysis covers the cities with tile-level density enrichment')
L('   (per_tile_enriched_all_cities.csv); cities without enrichment are excluded.')
L('   The full pipeline rerun restored coverage to ~135 vector / ~133 raster cities')
L('   (exact counts in Section 1).')
L('')
L('3. **SpaceNet vs other is a methodological split, not a quality split.**')
L('   SpaceNet7 reference data is highly accurate but covers a specific set of cities.')
L('   "Other" reference data varies in source and quality (WBG, HOT-OSM, local cadastre).')
L('   Higher variance in the "other" group should be expected.')
L('')
L('4. **Density/size classes are fixed.** The canonical breaks (density [0, 20, 100, 300, ∞]')
L('   bldg/km²; size [0, 80, 120, 180, ∞] m²) are applied identically to every subset and to')
L('   both vector and raster, so the figure families are directly comparable. They superseded')
L('   the earlier per-subset data-driven quartiles (2026-07-22). Every class is well populated')
L('   at tile level; at **city** level the sparsest density class (<20 bldg/km²) holds only')
L('   ~2 cities, so annotate n per class in any city-level table.')
L('')
L('5. **Raster density join.** Raster tiles are matched to density values using the same')
L('   tile grid as vector. Cities without density enrichment are dropped. The join % is')
L('   printed in cell 5 output.')
L('')
L('6. **LME convergence.** The mixed-effects model may not converge for small subsets')
L('   (e.g. spacenet-only). Check the statsmodels convergence warning in the cell output.')
L('   The LME uses density and size jointly as fixed effects — do not interpret the')
L('   marginal Spearman ρ and LME β for the same predictor as independent estimates.')
L('')

# ── Figures ────────────────────────────────────────────────────────────────────
L('---')
L('## 9. Output figures\n')
generated = sorted(FIGURES_DIR.glob('*.png'))
if generated:
    for f in generated:
        L(f'- `{f.name}`')
else:
    L('- No figures found in `outputs/figures/`.')

# ── Write file ─────────────────────────────────────────────────────────────────
SUMMARY_PATH.write_text('\n'.join(lines), encoding='utf-8')

print('=' * 65)
print(f'  Summary saved → {SUMMARY_PATH}')
print('=' * 65)
print()
print('\n'.join(lines))